In [2]:
# %pip install --upgrade azure-ai-ml --force-reinstall
# %pip install --upgrade mltable azureml-dataprep[pandas] --force-reinstall

  Using cached mltable-1.5.0-py3-none-any.whl (181 kB)
  Using cached azureml_dataprep-4.12.6-py3-none-any.whl (38.2 MB)
  Using cached PyYAML-6.0.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (705 kB)
  Using cached jsonschema-4.19.1-py3-none-any.whl (83 kB)
  Using cached msrest-0.7.1-py3-none-any.whl (85 kB)
  Using cached azure_core-1.29.5-py3-none-any.whl (192 kB)
  Using cached azure_mgmt_core-1.4.0-py3-none-any.whl (27 kB)
  Using cached python_dateutil-2.8.2-py2.py3-none-any.whl (247 kB)
  Using cached cryptography-41.0.5-cp37-abi3-manylinux_2_28_x86_64.whl (4.4 MB)
  Using cached PyJWT-2.8.0-py3-none-any.whl (22 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.5/502.5 kB 16.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.1/31.1 MB 43.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.7/191.7 kB 24.6 MB/s eta 0:00:00
  Using cached azureml_dataprep_rslex-2.19.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64

In [1]:
# Import required libraries
import os
from azure.identity import DefaultAzureCredential
from azure.identity import AzureCliCredential
from azure.ai.ml import automl, Input, MLClient, command

from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data
from azure.ai.ml.automl import (
    classification,
    ClassificationPrimaryMetrics,
    ClassificationModels,
)

In [2]:
# get handle on workspace
credential = DefaultAzureCredential()
ml_client = None
try:
    ml_client = MLClient.from_config(credential)
except Exception as ex:
    print(ex)
    # Enter details of your AML workspace
    subscription_id = "<SUBSCRIPTION_ID>"
    resource_group = "<RESOURCE_GROUP>"
    workspace = "<AML_WORKSPACE_NAME>"
    ml_client = MLClient(credential, subscription_id, resource_group, workspace)

Found the config file in: /config.json


In [3]:
workspace = ml_client.workspaces.get(name=ml_client.workspace_name)

subscription_id = ml_client.connections._subscription_id
resource_group = workspace.resource_group
workspace_name = ml_client.workspace_name

output = {}
output["Workspace"] = workspace_name
output["Subscription ID"] = subscription_id
output["Resource Group"] = resource_group
output["Location"] = workspace.location
output

{'Workspace': 'amlws-001',
 'Subscription ID': '0b5bb8c4-4e13-446b-b2d7-486015352171',
 'Resource Group': 'aiml-rg',
 'Location': 'eastus'}

In [4]:
## DATA
# create mltable yml file

import mltable
import os
import pandas as pd

from azure.ai.ml import MLClient
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential


# all_cols = [
#     'model', 'id', 'qcat', 'qid', 'question',
#     'w3_rated_gender', 'w3_rated_age', 'w3_rated_height', 'w3_rated_weight',
#     'w3_rated_race', 'w3_rated_income_19', 'w3_rated_education',
#     'w3_rated_conservative', 'w3_rated_voted', 'w3_rated_extravert',
#     'w3_rated_agreeable', 'w3_rated_concientious', 'w3_rated_neurotic',
#     'w3_rated_open', 'w3_rated_SDO_', 'w3_rated_trust',
#     'w3_rated_interact', 
#     'response',  'response_length', 'mean_word_length',
#     'qcat_label', 
#     'uses_you', 'uses_i', 'uses_we', 'uses_they',
#     'uses_exclamation', 'uses_question', 'uses_parentheses',
#     'uses_ellipsis', 'uses_all_caps',
#     'flesch_reading_ease','flesch_kincaid_grade', 'dale_chall_readability',
# ]

# cycle through all the target variables,
# load the data into mltable
# take an 80% sample for training
# drop columns other than target column and predictors
# create target subdirectory within training
# save mltable
targetVariables = [
    'w3_rated_gender', 'w3_rated_age', 'w3_rated_height', 'w3_rated_weight',
    'w3_rated_race', 'w3_rated_income_19', 'w3_rated_education',
    'w3_rated_conservative', 'w3_rated_voted', 'w3_rated_extravert',
    'w3_rated_agreeable', 'w3_rated_concientious', 'w3_rated_neurotic',
    'w3_rated_open', 'w3_rated_SDO_',
    ]

variablesToDrop = [
    'model', 'id', 'qcat', 'qid', 'question',
    'w3_rated_trust',
    'w3_rated_interact', 
    'response', 
    ]

df = pd.read_csv('./data/wave 3 response evals - long v4 - with qr text - for analysis.csv')

for targetv in targetVariables:

    print(f'working on {targetv}')

    # Microsoft.DPrep.DropColumnsBlock failed with NotImplementedError
    # so using pandas and relying on mltable as little as possible

    # copy the data using pandas
    dfv = df.copy()

    # drop rows where the target variable is missing
    dfv = dfv.loc[~dfv[targetv].isna(),].copy()

    # drop the columns
    thisSetDrop = variablesToDrop + [v for v in targetVariables if v != targetv]
    print('dropping ') 
    for v in thisSetDrop: print(v)

    # print('keeping ')
    # for v in [e for e in all_cols if e not in thisSetDrop]: print(v)

    dfv = dfv.drop(columns=thisSetDrop)

    # take random sample for training
    dfv = dfv.sample(frac=0.8, random_state=123).copy()

    # create training directory for this target variable
    os.makedirs(f'./data/train_{targetv}', exist_ok=True)
    os.makedirs(f'./data/train_{targetv}/data', exist_ok=True)

    # write the data to csv in data subfolder
    csv_path = f'./data/train_{targetv}/data/predict-{targetv}-train-data.csv'
    dfv.to_csv(csv_path, index=False)

    print(f'wrote dataframe keeping \n{dfv.columns}')

    paths = [{'file': csv_path}]

    # link to the data (in local path)
    tbl = mltable.from_delimited_files(paths)

    # this creates the MLTable yml file
    tbl.save(
        f'./data/train_{targetv}', 
        colocated=True, show_progress=True, overwrite=True)

    # now upload to cloud storage and bookmark

    # set VERSION variable
    VERSION="4"

    this_data = Data(
        path=f'./data/train_{targetv}',
        type=AssetTypes.MLTABLE,
        description=f"80% sample of inferred attribute {targetv} and predictors based on features of the textual response",
        name=f"predict-{targetv}-train-data",
        version=VERSION,
    )

    ml_client.data.create_or_update(this_data)

    print('\n\n')

working on w3_rated_gender
dropping 
model
id
qcat
qid
question
w3_rated_trust
w3_rated_interact
response
w3_rated_age
w3_rated_height
w3_rated_weight
w3_rated_race
w3_rated_income_19
w3_rated_education
w3_rated_conservative
w3_rated_voted
w3_rated_extravert
w3_rated_agreeable
w3_rated_concientious
w3_rated_neurotic
w3_rated_open
w3_rated_SDO_
wrote dataframe keeping 
Index(['w3_rated_gender', 'response_length', 'mean_word_length', 'qcat_label',
       'uses_you', 'uses_i', 'uses_we', 'uses_they', 'uses_exclamation',
       'uses_question',
       ...
       'online', 'or', 'out', 'outgoing', 'people', 'positive', 'priority',
       'probably', 'quite', 'read'],
      dtype='object', length=116)
Copying 1 files with concurrency set to 1
Copied /mnt/batch/tasks/shared/LS_root/mounts/clusters/e4-ds-v4-a/code/Users/niswitan/ai-responses/data/train_w3_rated_gender/data/predict-w3_rated_gender-train-data.csv, file 1 out of 1. Destination path: /mnt/batch/tasks/shared/LS_root/mounts/clusters

Uploading train_w3_rated_gender (35.86 MBs): 100%|██████████| 35860725/35860725 [00:00<00:00, 122186744.10it/s]


Uploading train_w3_rated_age (36.51 MBs): 100%|██████████| 36512935/36512935 [00:00<00:00, 105950051.35it/s]


Uploading train_w3_rated_height (36.1 MBs): 100%|██████████| 36101415/36101415 [00:00<00:00, 103139902.26it/s]


Uploading train_w3_rated_weight (36.3 MBs): 100%|██████████| 36302405/36302405 [00:00<00:00, 93934562.18it/s] 


Uploading train_w3_rated_race (36.45 MBs): 100%|██████████| 36452097/36452097 [00:00<00:00, 84595473.77it/s]


Uploading train_w3_rated_income_19 (36.48 MBs): 100%|██████████| 36483971/36483971 [00:00<00:00, 110033073.23it/s]


Uploading train_w3_rated_education (36.45 MBs): 100%|██████████| 36446669/36446669 [00:00<00:00, 117191820.77it/s]


Uploading train_w3_rated_conservative (36.45 MBs): 100%|██████████| 36446825/36446825 [00:00<00:00, 108429849.20it/s]


Uploading train_w3_rated_voted (36.42 MBs): 100%|██████████| 36423929/36423929 [00:0

In [26]:
# testing that this works

# # set VERSION variable
# VERSION="1"

# this_data = Data(
#     path=f'./data/train_{targetv}',
#     type=AssetTypes.MLTABLE,
#     description=f"80% sample of inferred attribute {targetv} and predictors based on features of the textual response",
#     name=f"predict-{targetv}-train-data",
#     version=VERSION,
# )

# ml_client.data.create_or_update(this_data)

Uploading train_w3_rated_gender (18.24 MBs): 100%|██████████| 18243207/18243207 [00:00<00:00, 60945543.19it/s]




Data({'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': ['./data/wave 3 response evals - long v4 - with qr text - for analysis.csv'], 'type': 'mltable', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'predict-w3_rated_gender-train-data', 'description': '80% sample of inferred attribute w3_rated_gender and predictors based on features of the textual response', 'tags': {}, 'properties': {}, 'print_as_yaml': True, 'id': '/subscriptions/0b5bb8c4-4e13-446b-b2d7-486015352171/resourceGroups/aiml-rg/providers/Microsoft.MachineLearningServices/workspaces/amlws-001/data/predict-w3_rated_gender-train-data/versions/1', 'Resource__source_path': None, 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/e4-ds-v4-a/code/Users/niswitan/ai-responses', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x7f69819bf7c0>, 'serialize': <msrest.serialization.Serializer object at 0x7f69819bd390>, 'version': 

In [5]:
# configure cluster
from azure.ai.ml.entities import AmlCompute
from azure.core.exceptions import ResourceNotFoundError

compute_name = "cpu-ds12-v2-8node"

try:
    _ = ml_client.compute.get(compute_name)
    print("Found existing compute target.")
except ResourceNotFoundError:
    print("Creating a new compute target...")
    compute_config = AmlCompute(
        name=compute_name,
        type="amlcompute",
        size="STANDARD_DS12_V2",
        idle_time_before_scale_down=1200,
        min_instances=0,
        max_instances=8,
    )
    ml_client.begin_create_or_update(compute_config).result()

Found existing compute target.


In [6]:
print(compute_name)
print(VERSION)

cpu-ds12-v2-8node
4


## Configure AutoML

In [7]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import automl, Input


targetVariables = [
    'w3_rated_gender', 'w3_rated_age', 'w3_rated_height', 'w3_rated_weight',
    'w3_rated_race', 'w3_rated_income_19', 'w3_rated_education',
    'w3_rated_conservative', 'w3_rated_voted', 'w3_rated_extravert',
    'w3_rated_agreeable', 'w3_rated_concientious', 'w3_rated_neurotic',
    'w3_rated_open', 'w3_rated_SDO_',
    ]

regressionTargetVariables = [
    'w3_rated_age', 'w3_rated_height', 
    'w3_rated_income_19', 
    ]

for targetv in targetVariables:
    print(f'configuring and submitting prediction model for {targetv}\n\n')

    # make an Input object for the training data
    data_asset = ml_client.data.get(
        name=f"predict-{targetv}-train-data", version=VERSION)

    my_training_data_input = Input(
        type=AssetTypes.MLTABLE, 
        path=data_asset.id,
    )

    my_exp_name = 'predict_' + targetv

    if targetv in regressionTargetVariables:
        # configure regression job
        # configure the classification job
        automl_job = automl.regression(
            compute=compute_name,
            experiment_name=my_exp_name,
            training_data=my_training_data_input,
            target_column_name=targetv,
            primary_metric="r2_score",
            # n_cross_validations=5,
            enable_model_explainability=True,
            tags={"tag": "predict attribute inference based on text features"}
        )

        # Training properties are optional
        automl_job.set_training(
            # blocked_training_algorithms=["logistic_regression"], 
            enable_onnx_compatible_models=True
        )

    else:
        # configure the classification job
        automl_job = automl.classification(
            compute=compute_name,
            experiment_name=my_exp_name,
            training_data=my_training_data_input,
            target_column_name=targetv,
            primary_metric="accuracy",
            # n_cross_validations=5,
            enable_model_explainability=True,
            tags={"tag": "predict attribute inference based on text features"}
        )

        # Training properties are optional
        automl_job.set_training(
            blocked_training_algorithms=["logistic_regression"], 
            enable_onnx_compatible_models=True
        )


    # set limits for either classification or regression job
    # Limits are all optional
    automl_job.set_limits(
        timeout_minutes=600, 
        trial_timeout_minutes=20, 
        max_trials=20,
        max_concurrent_trials=8,
        max_cores_per_trial=-1,
        enable_early_termination=True,
    )


    # Submit the AutoML job
    returned_job = ml_client.jobs.create_or_update(
        automl_job
        )  # submit the job to the backend

    print(f"Created job: {returned_job}")

configuring and submitting prediction model for w3_rated_gender


Created job: compute: azureml:cpu-ds12-v2-8node
creation_context:
  created_at: '2023-10-29T22:02:29.585272+00:00'
  created_by: Nick Switanek
  created_by_type: User
display_name: olden_loquat_6zzwrg3hzx
experiment_name: predict_w3_rated_gender
id: azureml:/subscriptions/0b5bb8c4-4e13-446b-b2d7-486015352171/resourceGroups/aiml-rg/providers/Microsoft.MachineLearningServices/workspaces/amlws-001/jobs/olden_loquat_6zzwrg3hzx
limits:
  enable_early_termination: true
  max_concurrent_trials: 8
  max_cores_per_trial: -1
  max_nodes: 1
  max_trials: 20
  timeout_minutes: 600
  trial_timeout_minutes: 20
log_verbosity: info
name: olden_loquat_6zzwrg3hzx
outputs: {}
primary_metric: accuracy
properties: {}
resources:
  instance_count: 1
  shm_size: 2g
services:
  Studio:
    endpoint: https://ml.azure.com/runs/olden_loquat_6zzwrg3hzx?wsid=/subscriptions/0b5bb8c4-4e13-446b-b2d7-486015352171/resourcegroups/aiml-rg/workspaces/amlws-0

Created job: compute: azureml:cpu-ds12-v2-8node
creation_context:
  created_at: '2023-10-29T05:09:31.211109+00:00'
  created_by: Nick Switanek
  created_by_type: User
display_name: jovial_berry_9wsd9635kp
experiment_name: predict_w3_rated_gender
id: azureml:/subscriptions/0b5bb8c4-4e13-446b-b2d7-486015352171/resourceGroups/aiml-rg/providers/Microsoft.MachineLearningServices/workspaces/amlws-001/jobs/jovial_berry_9wsd9635kp
limits:
  enable_early_termination: true
  max_concurrent_trials: 8
  max_cores_per_trial: -1
  max_nodes: 1
  max_trials: 4
  timeout_minutes: 60
  trial_timeout_minutes: 20
log_verbosity: info
name: jovial_berry_9wsd9635kp
outputs: {}
primary_metric: accuracy
properties: {}
resources:
  instance_count: 1
  shm_size: 2g
services:
  Studio:
    endpoint: https://ml.azure.com/runs/jovial_berry_9wsd9635kp?wsid=/subscriptions/0b5bb8c4-4e13-446b-b2d7-486015352171/resourcegroups/aiml-rg/workspaces/amlws-001&tid=16b3c013-d300-468d-ac64-7eda0820b6d3
  Tracking:
    endpoint